# MLflow + Hyperopt — Training & Tracking an ANN Model
Channel: Asadullah AI

Prerequisite: Basic understanding of Machine Learning and Artificial Neural Networks (ANN).

### What you will learn

- Build a Keras Sequential ANN for Wine Quality prediction
- Apply normalization using mean & variance
- Run Hyperopt for hyperparameter search (learning rate, momentum)
- Track every experiment in MLflow (nested runs)
- Register the best model in the Model Registry
- Load and inference using PyFunc

In [13]:
import keras
import numpy as np
import pandas as pd
from hyperopt import STATUS_OK,Trials,fmin,hp,tpe
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [29]:
mlflow.set_tracking_uri(uri="http://127.0.0.1:5000/")

In [30]:
## load the dataset
data=pd.read_csv(
    "https://raw.githubusercontent.com/mlflow/mlflow/master/tests/datasets/winequality-white.csv",
    sep=";",
)
data

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
0,7.0,0.27,0.36,20.7,0.045,45.0,170.0,1.00100,3.00,0.45,8.8,6
1,6.3,0.30,0.34,1.6,0.049,14.0,132.0,0.99400,3.30,0.49,9.5,6
2,8.1,0.28,0.40,6.9,0.050,30.0,97.0,0.99510,3.26,0.44,10.1,6
3,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
4,7.2,0.23,0.32,8.5,0.058,47.0,186.0,0.99560,3.19,0.40,9.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4893,6.2,0.21,0.29,1.6,0.039,24.0,92.0,0.99114,3.27,0.50,11.2,6
4894,6.6,0.32,0.36,8.0,0.047,57.0,168.0,0.99490,3.15,0.46,9.6,5
4895,6.5,0.24,0.19,1.2,0.041,30.0,111.0,0.99254,2.99,0.46,9.4,6
4896,5.5,0.29,0.30,1.1,0.022,20.0,110.0,0.98869,3.34,0.38,12.8,7


In [31]:
## Split the data into training,validation and test sets

train,test=train_test_split(data,test_size=0.25,random_state=42)
train

,fixed acidity,volatile acidity,citric acid,residual sugar,chlorides,free sulfur dioxide,total sulfur dioxide,density,pH,sulphates,alcohol,quality
2835,6.3,0.25,0.22,3.30,0.048,41.0,161.0,0.99256,3.16,0.50,10.5,6
1157,7.8,0.30,0.29,16.85,0.054,23.0,135.0,0.99980,3.16,0.38,9.0,6
744,7.4,0.38,0.27,7.50,0.041,24.0,160.0,0.99535,3.17,0.43,10.0,5
1448,7.4,0.16,0.49,1.20,0.055,18.0,150.0,0.99170,3.23,0.47,11.2,6
3338,7.2,0.27,0.28,15.20,0.046,6.0,41.0,0.99665,3.17,0.39,10.9,6
...,...,...,...,...,...,...,...,...,...,...,...,...
4426,6.2,0.21,0.52,6.50,0.047,28.0,123.0,0.99418,3.22,0.49,9.9,6
466,7.0,0.14,0.32,9.00,0.039,54.0,141.0,0.99560,3.22,0.43,9.4,6
3092,7.6,0.27,0.52,3.20,0.043,28.0,152.0,0.99129,3.02,0.53,11.4,6
3772,6.3,0.24,0.29,13.70,0.035,53.0,134.0,0.99567,3.17,0.38,10.6,6


In [32]:
train[['quality']].values.ravel()

array([6, 6, 5, ..., 6, 6, 8], shape=(3673,))

In [33]:
train_x=train.drop(['quality'],axis=1).values
train_y=train[['quality']].values.ravel()

## test dataset
test_x=test.drop(['quality'],axis=1).values
test_y=test[['quality']].values.ravel()

## splitting this train data into train and validation

train_x,valid_x,train_y,valid_y=train_test_split(train_x,train_y,test_size=0.20,random_state=42)

signature=infer_signature(train_x,train_y)

In [34]:
np.mean(train_x,axis=0)

array([6.86621852e+00, 2.80377808e-01, 3.32597005e-01, 6.42164738e+00,
       4.55513955e-02, 3.53556841e+01, 1.38792376e+02, 9.94074221e-01,
       3.18919333e+00, 4.88396869e-01, 1.05005673e+01])

In [35]:
train_x.shape[1]

11

In [36]:
### ANN Model

def train_model(params,epochs,train_x,train_y,valid_x,valid_y,test_x,test_y):

    ## Define model architecture
    mean=np.mean(train_x,axis=0)
    var=np.var(train_x,axis=0)

    model=keras.Sequential(
        [
            keras.Input([train_x.shape[1]]),
            keras.layers.Normalization(mean=mean,variance=var),
            keras.layers.Dense(64,activation='relu'),
            keras.layers.Dense(1)

        ]
    )

    ## compile the model
    model.compile(
        optimizer=keras.optimizers.SGD(
        learning_rate=params["lr"],
        momentum=params["momentum"]
    ),
    loss="mean_squared_error",
    metrics=[keras.metrics.RootMeanSquaredError()]
    )

    ## Train the ANN model with lr and momentum params wwith MLFLOW tracking
    with mlflow.start_run(nested=True):
        model.fit(train_x,train_y,validation_data=(valid_x,valid_y),
                  epochs=epochs,
                  batch_size=64)
        
        ## Evaluate the model
        eval_result=model.evaluate(valid_x,valid_y,batch_size=64)

        eval_rmse=eval_result[1]

        ## Log the parameters and results
        mlflow.log_params(params)
        mlflow.log_metric("eval_rmse",eval_rmse)

        ## log the model

        mlflow.tensorflow.log_model(model,"ann-model",signature=signature)

        return {"loss": eval_rmse, "status": STATUS_OK, "model": model}

In [37]:
def objective(params):
    # MLflow will track the parameters and results for each run
    result = train_model(
        params,
        epochs=3,
        train_x=train_x,
        train_y=train_y,
        valid_x=valid_x,
        valid_y=valid_y,
        test_x=test_x,
        test_y=test_y,
    )
    return result

In [38]:
space={
    "lr":hp.loguniform("lr",np.log(1e-5),np.log(1e-1)),
    "momentum":hp.uniform("momentum",0.0,1.0)

}

In [39]:
mlflow.set_experiment("wine-quality")
with mlflow.start_run():
    # Conduct the hyperparameter search using Hyperopt
    trials=Trials()
    best=fmin( 
        fn=objective,
        space=space,
        algo=tpe.suggest, # Tree-structured parzen estimator 
        max_evals=4,
        trials=trials
    )

    # Fetch the details of the best run
    best_run = sorted(trials.results, key=lambda x: x["loss"])[0]

    # Log the best parameters, loss, and model
    mlflow.log_params(best)
    mlflow.log_metric("eval_rmse", best_run["loss"])
    mlflow.tensorflow.log_model(best_run["model"], "ann-best-model", signature=signature)

    # Print out the best parameters and corresponding loss
    print(f"Best parameters: {best}")
    print(f"Best eval rmse: {best_run['loss']}")


2026/09/07 04:12:39 INFO mlflow.tracking.fluent: Experiment with name 'wine-quality' does not exist. Creating a new experiment.


Epoch 1/3                                            

 1/46 ━━━━━━━━━━━━━━━━━━━━ 6:25 9s/step - loss: 44.0197 - root_mean_squared_error: 6.6347
 2/46 ━━━━━━━━━━━━━━━━━━━━ 2s 62ms/step - loss: 37.0522 - root_mean_squared_error: 6.0598
 3/46 ━━━━━━━━━━━━━━━━━━━━ 3s 81ms/step - loss: 31.9887 - root_mean_squared_error: 5.5984
 4/46 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - loss: 28.2201 - root_mean_squared_error: 5.2270
 5/46 ━━━━━━━━━━━━━━━━━━━━ 2s 72ms/step - loss: 25.3676 - root_mean_squared_error: 4.9288
 6/46 ━━━━━━━━━━━━━━━━━━━━ 2s 67ms/step - loss: 23.1850 - root_mean_squared_error: 4.6912
 7/46 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - loss: 21.4143 - root_mean_squared_error: 4.4903
 8/46 ━━━━━━━━━━━━━━━━━━━━ 2s 78ms/step - loss: 19.9397 - root_mean_squared_error: 4.3166
 9/46 ━━━━━━━━━━━━━━━━━━━━ 2s 75ms/step - loss: 18.6976 - root_mean_squared_error: 4.1659
10/46 ━━━━━━━━━━━━━━━━━━━━ 2s 77ms/step - loss: 17.6325 - root_mean_squared_error: 4.0330
11/46 ━━━━━━━━━━━━━━━━━━━━ 2s 79ms/step - los

2026/09/07 04:13:14 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run honorable-whale-535 at: http://127.0.0.1:5000/#/experiments/1/runs/dbe8fe3494b3403f92e6d39184fd91a1

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 2:53 4s/step - loss: 38.2399 - root_mean_squared_error: 6.1838
 9/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 28.1599 - root_mean_squared_error: 5.2701 
11/46 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 26.0099 - root_mean_squared_error: 5.0466
14/46 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 23.4677 - root_mean_squared_error: 4.7710
16/46 ━━━━━━━━━━━━━━━━━━━━ 0s 19ms/step - loss: 22.1074 - root_mean_squared_error: 4.6180
17/46 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step - loss: 21.5004 - root_mean_squared_error: 4.5483
19/46 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 20.3926 - root_mean_squared_error: 4.4183
20/46 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 19.8851 - root_mean_squared_error: 4.3574
23/46 ━━━━━━━━━━━━━━━━━━━━

2026/09/07 04:14:10 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



...

🏃 View run magnificent-fawn-897 at: http://127.0.0.1:5000/#/experiments/1/runs/8d3662b53f4740ee8d9ebdead9cf284c

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

Epoch 1/3                                                                      

 1/46 ━━━━━━━━━━━━━━━━━━━━ 1:18 2s/step - loss: 29.2208 - root_mean_squared_error: 5.4056
 8/46 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 30.5159 - root_mean_squared_error: 5.5239 
13/46 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 30.2308 - root_mean_squared_error: 5.4980
14/46 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 30.1878 - root_mean_squared_error: 5.4941
18/46 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 29.9718 - root_mean_squared_error: 5.4743
23/46 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 29.6569 - root_mean_squared_error: 5.4453
29/46 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 29.2268 - root_mean_squared_error: 5.4052
36/46 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 28.6887 - root_mean_squared_error: 5.3543
42/46 

2026/09/07 04:15:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.



🏃 View run flawless-crane-834 at: http://127.0.0.1:5000/#/experiments/1/runs/3e786766737142e2b7f8c63050def71c

🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1                   

100%|██████████| 4/4 [03:13<00:00, 48.35s/trial, best loss: 0.7447970509529114]


2026/09/07 04:15:53 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.


Best parameters: {'lr': np.float64(0.03639081856402229), 'momentum': np.float64(0.2021902157185983)}
Best eval rmse: 0.7447970509529114
🏃 View run casual-hog-592 at: http://127.0.0.1:5000/#/experiments/1/runs/027ba5d011e24b1fa4df129c41d5ac08
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/1


In [45]:
# Load model as a PyFuncModel.
model_uri = 'models:/m-1c10f04e5d4e45f9bc47d3c1bcd3e7ab/'
loaded_model = mlflow.pyfunc.load_model(model_uri)

# Predict on a Pandas DataFrame.
import pandas as pd
loaded_model.predict(pd.DataFrame(test_x))

39/39 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step


array([[6.2615747],
       [6.6557026],
       [6.3611484],
       ...,
       [6.048317 ],
       [6.560148 ],
       [5.6048846]], shape=(1225, 1), dtype=float32)